In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.firefox.service import Service as FirefoxService
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.firefox import GeckoDriverManager

# The scrape_cricbuzz_commentary function from the previous answer goes here.
# It does not need any changes.
def scrape_cricbuzz_commentary(url):
    """
    Scrapes structured, ball-by-ball commentary from a Cricbuzz match URL.

    This function uses a single, robust strategy:
    1. Loads the page (assumes commentary is default).
    2. Clicks the "Load More Commentary" button until it disappears.
    3. Scrapes the full commentary data into a structured format.
    """
    print(f"--- Starting scrape for Cricbuzz URL: {url} ---")

    service = FirefoxService(GeckoDriverManager().install())
    driver = webdriver.Firefox(service=service)
    wait = WebDriverWait(driver, 10)
    
    commentary_data = []

    try:
        # Step 1: Load the page
        driver.get(url)
        # Wait for the first commentary number to be present before proceeding
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "cb-ovr-num")))
        time.sleep(1)

        # --- Step 2: Click the "Load More" button until the entire commentary is visible ---
        print("Expanding all commentary by clicking the 'Load More' button...")
        while True:
            try:
                # Find the button by its unique ID. A short wait is sufficient.
                load_more_button = WebDriverWait(driver, 3).until(
                    EC.element_to_be_clickable((By.ID, "full_commentary_btn"))
                )
                driver.execute_script("arguments[0].click();", load_more_button)
                print("    ...clicked 'Load More'.")
                time.sleep(2) # Wait for new comments to load
            except (TimeoutException, NoSuchElementException):
                # This is the expected way to exit the loop: the button is no longer on the page.
                print("  'Load More' button no longer found. All commentary is now visible.")
                break
            except Exception as e:
                print(f"  An unexpected error occurred while clicking 'Load More': {e}")
                break

        # --- Step 3: Scrape the structured commentary ---
        print("\nScraping structured ball-by-ball data...")

        # Find the parent container for each ball's commentary.
        # This robust XPath finds divs that contain 'cb-col' in their class AND
        # also contain the specific over number element, ensuring we only get commentary blocks.
        commentary_containers = driver.find_elements(By.XPATH, "//div[contains(@class, 'cb-col') and .//div[contains(@class, 'cb-ovr-num')]]")

        if not commentary_containers:
            print("No commentary containers were found after loading. Check XPath and page structure.")
        else:
            for container in commentary_containers:
                try:
                    over = container.find_element(By.CLASS_NAME, 'cb-ovr-num').text.strip()
                    text = container.find_element(By.CLASS_NAME, 'cb-com-ln').text.strip()
                    if over and text:
                        commentary_data.append({"over": over, "commentary": text})
                    elif text: #This usually means there's no commentary present
                        commentary_data.append({"commentary": text})
                except NoSuchElementException:
                    # Gracefully skip any block that doesn't fit the structure (e.g., ads, "End of Over")
                    pass
            print(f"Successfully scraped {len(commentary_data)} ball-by-ball entries.")

    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    finally:
        # It's important to quit the driver after each URL to free up resources
        driver.quit()

    return commentary_data

# --- [MODIFIED] How to use the function in a loop ---
if __name__ == "__main__":
    # 1. Define your list of URLs to scrape
    urls_to_scrape = [
        "https://www.cricbuzz.com/cricket-scores/32287/ind-vs-eng-1st-odi-england-tour-of-india-2021",
        "https://www.cricbuzz.com/cricket-scores/32292/ind-vs-eng-2nd-odi-england-tour-of-india-2021",
        "https://www.cricbuzz.com/cricket-scores/32293/ind-vs-eng-3rd-odi-england-tour-of-india-2021",
        "https://www.cricbuzz.com/cricket-scores/2188/ind-vs-ire-22nd-match-group-b-icc-world-cup-2011",
        "https://www.cricbuzz.com/cricket-scores/2191/ind-vs-ned-25th-match-group-b-icc-world-cup-2011",
        "https://www.cricbuzz.com/cricket-scores/112455/nz-vs-ind-12th-match-group-a-icc-champions-trophy-2025",
        "https://www.cricbuzz.com/cricket-scores/112469/ind-vs-nz-final-icc-champions-trophy-2025",
        "https://www.cricbuzz.com/cricket-scores/112420/pak-vs-ind-5th-match-group-a-icc-champions-trophy-2025",
        "http://cricbuzz.com/cricket-scores/2195/ind-vs-rsa-29th-match-group-b-icc-world-cup-2011",
        "https://www.cricbuzz.com/cricket-scores/36526/sl-vs-ind-1st-odi-india-tour-of-sri-lanka-2021",
        "https://www.cricbuzz.com/cricket-scores/36531/sl-vs-ind-2nd-odi-india-tour-of-sri-lanka-2021",
        "https://www.cricbuzz.com/cricket-scores/36536/sl-vs-ind-3rd-odi-india-tour-of-sri-lanka-2021",
        "https://www.cricbuzz.com/cricket-scores/10799/ire-vs-nam-1st-match-icc-intercontinental-cup-2011",
        "https://www.cricbuzz.com/cricket-scores/35172/ned-vs-ire-1st-odi-ireland-tour-of-netherlands-2021",
        "https://www.cricbuzz.com/cricket-scores/35177/ned-vs-ire-2nd-odi-ireland-tour-of-netherlands-2021",
        "https://www.cricbuzz.com/cricket-scores/35178/ned-vs-ire-3rd-odi-ireland-tour-of-netherlands-2021",
        "https://www.cricbuzz.com/cricket-scores/2203/ire-vs-ned-37th-match-group-b-icc-world-cup-2011",
        "https://www.cricbuzz.com/cricket-scores/35127/ire-vs-rsa-2nd-odi-south-africa-tour-of-ireland-2021",
        "https://www.cricbuzz.com/cricket-scores/2193/wi-vs-ire-27th-match-group-b-icc-world-cup-2011",
        "https://www.cricbuzz.com/cricket-scores/3334/wi-vs-ire-only-odi-ireland-in-west-indies-odi-match",
        "https://www.cricbuzz.com/cricket-scores/2197/aus-vs-ken-31st-match-group-a-icc-world-cup-2011",
        "https://www.cricbuzz.com/cricket-scores/2189/can-vs-ken-23rd-match-group-a-icc-world-cup-2011",
        "https://www.cricbuzz.com/cricket-scores/3310/ken-vs-ned-1st-odi-netherlands-in-kenya-odi-series",
        "https://www.cricbuzz.com/cricket-scores/3169/zim-vs-ken-1st-odi-kenya-in-zimbabwe-odi-series",
        "https://www.cricbuzz.com/cricket-scores/2198/ban-vs-ned-32nd-match-group-a-icc-world-cup-2011",
        "https://www.cricbuzz.com/cricket-scores/10734/sco-vs-ned-1st-odi-icc-intercontinental-cup-one-day-2011-2013",
        "https://www.cricbuzz.com/cricket-scores/3168/aus-vs-nz-final-icc-champions-trophy-2009",
        "https://www.cricbuzz.com/cricket-scores/30885/nz-vs-ban-1st-odi-bangladesh-tour-of-new-zealand-2021",
        "https://www.cricbuzz.com/cricket-scores/30890/nz-vs-ban-2nd-odi-bangladesh-tour-of-new-zealand-2021",
        "https://www.cricbuzz.com/cricket-scores/30894/nz-vs-ban-3rd-odi-bangladesh-tour-of-new-zealand-2021",
        "https://www.cricbuzz.com/cricket-scores/112427/ban-vs-nz-6th-match-group-a-icc-champions-trophy-2025",
        "https://www.cricbuzz.com/cricket-scores/2196/nz-vs-can-30th-match-group-a-icc-world-cup-2011",
        "https://www.cricbuzz.com/cricket-scores/112395/pak-vs-nz-1st-match-group-a-icc-champions-trophy-2025",
        "https://www.cricbuzz.com/cricket-scores/3179/nz-vs-pak-1st-odi-pakistan-v-new-zealand-odi-series",
        "https://www.cricbuzz.com/cricket-scores/3180/nz-vs-pak-2nd-odi-pakistan-v-new-zealand-odi-series",
        "https://www.cricbuzz.com/cricket-scores/3181/nz-vs-pak-3rd-odi-pakistan-v-new-zealand-odi-series",
        # Add as many URLs as you want here
    ]

    # 2. Create an empty list to store all the data from all matches
    all_matches_data = []

    # 3. Loop through each URL
    for url in urls_to_scrape:
        print(f"\n{'='*20} PROCESSING NEW MATCH {'='*20}")
        # Call the scraping function for the current URL
        single_match_list = scrape_cricbuzz_commentary(url)

        # 4. Check if data was successfully scraped for this match
        if single_match_list:
            # BEST PRACTICE: Add a column to identify the source of the data
            for entry in single_match_list:
                entry['source_url'] = url
            
            # Add the data from this match to our master list
            all_matches_data.extend(single_match_list)
            print(f"--- Added {len(single_match_list)} entries to the main dataset ---")
        else:
            print(f"--- No data scraped for {url}, moving to the next one. ---")
        
        # Polite pause between requests to avoid overwhelming the server
        time.sleep(5) 

    # 5. After the loop, check if any data was collected at all
    if all_matches_data:
        # Convert the complete list of dictionaries into one final DataFrame
        combined_df = pd.DataFrame(all_matches_data)
        
        output_filename = "cricbuzz_all_matches_commentary.csv"
        combined_df.to_csv(output_filename, index=False, encoding='utf-8')
        
        print(f"\n{'='*20} SCRAPING COMPLETE {'='*20}")
        print(f"All data saved to '{output_filename}'")
        print(f"Total entries scraped from all matches: {len(combined_df)}")
        print("\n--- First 5 rows of the combined DataFrame ---")
        print(combined_df.head().to_string())
        print("\n--- Last 5 rows of the combined DataFrame ---")
        print(combined_df.tail().to_string())
    else:
        print("\n--- Scraping finished, but no data was collected from any of the URLs. ---")


==================== PROCESSING NEW MATCH ====================
--- Starting scrape for Cricbuzz URL: https://www.cricbuzz.com/cricket-scores/32287/ind-vs-eng-1st-odi-england-tour-of-india-2021 ---
Looking for a 'Load More' button to click...
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
    ...clicked 'Load More'.
 